EMAIL SPAM DETECTION USING MACHINE LEARNING

Import libraries

In [38]:
import pandas as pd
import numpy as np

In [39]:
import warnings
warnings.filterwarnings("ignore")


In [40]:
from google.colab import files
uploaded=files.upload()

Saving combined_dataset.csv to combined_dataset (1).csv


In [41]:
df=pd.read_csv("combined_dataset.csv")

Exploring dataset

In [42]:
print(df.head())

  target                                               text
0   spam  Congratulations! You've been selected for a lu...
1   spam  URGENT: Your account has been compromised. Cli...
2   spam  You've won a free iPhone! Claim your prize by ...
3   spam  Act now and receive a 50% discount on all purc...
4   spam  Important notice: Your subscription will expir...


In [43]:
print(df.shape)

(10961, 2)


In [44]:
print(df.tail())

      target                                               text
10956   spam  Hey little one! Exciting news! Mama and baby a...
10957   spam  Amazing DATA deals on your Pulse Plan today! D...
10958   spam  Special offer just for you! Get 1GB @15 bob va...
10959   spam  NEW ARRIVAL - JUNE 23RD  Dresses @ 300; Kondel...
10960   spam  Coureen, did you know that saving on Timiza in...


In [45]:
df.columns.tolist()

['target', 'text']

In [46]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10961 entries, 0 to 10960
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   target  10961 non-null  object
 1   text    10961 non-null  object
dtypes: object(2)
memory usage: 171.4+ KB


In [47]:
print(df.isnull().sum())

target    0
text      0
dtype: int64


In [48]:
df.duplicated().sum()

np.int64(674)

Data Cleaning & Target Encoding

In [49]:
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)

In [50]:
# Remove empty text rows
df = df[df["text"].str.strip() != ""]
df.reset_index(drop=True, inplace=True)

In [51]:
df.shape

(10286, 2)

In [52]:
# Encode target:
df["target"] = df["target"].map({"ham": 0, "spam": 1})

In [53]:
print(df["target"].value_counts())

target
0    8014
1    2272
Name: count, dtype: int64


In [54]:
print(df.head())


   target                                               text
0       1  Congratulations! You've been selected for a lu...
1       1  URGENT: Your account has been compromised. Cli...
2       1  You've won a free iPhone! Claim your prize by ...
3       1  Act now and receive a 50% discount on all purc...
4       1  Important notice: Your subscription will expir...


In [55]:
print(df.isnull().sum())

target    0
text      0
dtype: int64


TF-IDF Vectorization & Train-Test Split

In [56]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [57]:
# Features and target
X = df["text"]
y = df["target"]


In [58]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [59]:
tfidf = TfidfVectorizer(stop_words="english", max_features=3000,lowercase=True)

In [60]:
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

Train Machine Learning Models

In [61]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [62]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42)
}


In [72]:
trained_models = {}

In [73]:
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    trained_models[name] = model

Compare Model Performance

In [64]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix


In [74]:
results = []
for name, model in trained_models.items():
    y_pred = model.predict(X_test_tfidf)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1 Score": f1
    })

results_df = pd.DataFrame(results).sort_values(by="F1 Score", ascending=False)

In [76]:
best_model_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_model_name]
print(f"\nBest Model: {best_model_name}")


Best Model: Random Forest


Save the Best Model & Vectorizer

In [77]:
import pickle

In [78]:
# Save the best model
with open("spam_model.pkl", "wb") as f:
    pickle.dump(best_model, f)

# Save the TF-IDF vectorizer
with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

In [79]:
print(f"Saved best model: {best_model_name}")

Saved best model: Random Forest


reload and verify it works

In [80]:
with open("spam_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

with open("tfidf_vectorizer.pkl", "rb") as f:
    loaded_vectorizer = pickle.load(f)

In [81]:
sample_text = ["Congratulations! You have won a free prize, click here now"]
sample_tfidf = loaded_vectorizer.transform(sample_text)
prediction = loaded_model.predict(sample_tfidf)

In [82]:
print("\nSanity Check Prediction:")
print("Message:", sample_text[0])
print("Prediction:", "Spam" if prediction[0] == 1 else "Ham")


Sanity Check Prediction:
Message: Congratulations! You have won a free prize, click here now
Prediction: Spam


In [84]:
def predict_message(message):
    message_tfidf = tfidf.transform([message])
    prediction = best_model.predict(message_tfidf)[0]
    return "Spam" if prediction == 1 else "Ham"


msg = input("Enter a message: ")
print(predict_message(msg))

Enter a message: Hey, are we still meeting for lunch tomorrow at 1pm? Let me know if that time works for you.
Ham
